# OpenAI Function Calling In LangChain

In [ ]:
# 读取 .env 里的 OPENAI_API_KEY
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

In [ ]:
# pydantic 用来定义"带类型校验"的数据模型，是后面把 Python 类自动转成 OpenAI function schema 的基础
from typing import List
from pydantic import BaseModel, Field

Pydantic Syntax

In [ ]:
# 普通 Python 类：完全没有类型校验，传什么进去就存什么，不会主动帮你检查类型是否正确
class User:
    def __init__(self, name: str, age: int, email: str):
        self.name = name
        self.age = age
        self.email = email

In [ ]:
foo = User(name="Joe",age=32, email="joe@gmail.com")

In [ ]:
foo.name

In [ ]:
# 注意：age 传的是字符串 "bar" 而不是 int，但普通类不会报错——这正是要和下面 pydantic 的行为做对比
foo = User(name="Joe",age="bar", email="joe@gmail.com")

In [ ]:
# foo.age 就是字符串 "bar"，普通类完全不会帮你做类型检查
foo.age

In [ ]:
# pydantic 模型：继承 BaseModel，并且用类型注解声明字段类型，pydantic 会在实例化时自动做类型校验/转换
class pUser(BaseModel):
    name: str
    age: int
    email: str

In [ ]:
foo_p = pUser(name="Jane", age=32, email="jane@gmail.com")

In [ ]:
foo_p.name

In [ ]:
# 这里故意传一个非法的 age="bar"，pydantic 会主动抛出 ValidationError（这是预期中的报错，用来演示类型校验）
foo_p = pUser(name="Jane", age="bar", email="jane@gmail.com")

In [ ]:
# pydantic 模型可以嵌套：这里 Class 里的 students 字段是一个 pUser 列表
class Class(BaseModel):
    students: List[pUser]

In [ ]:
obj = Class(
    students=[pUser(name="Jane", age=32, email="jane@gmail.com")]
)

In [ ]:
obj

## Pydantic to OpenAI function definition

In [ ]:
# 用 pydantic 模型描述一个"函数"：类的 docstring 会变成 function 的 description，
# Field(description=...) 会变成对应参数的 description，这些都是 OpenAI function schema 需要的信息
class WeatherSearch(BaseModel):
    """Call this with an airport code to get the weather at that airport"""
    airport_code: str = Field(description="airport code to get weather for")

In [ ]:
# 【版本兼容修复】原来的 langchain.utils.openai_functions 模块已经不存在了（langchain.utils 整个没了），
# convert_pydantic_to_openai_function 这个工具函数现在放在 langchain_community.utils.openai_functions 下
from langchain_community.utils.openai_functions import convert_pydantic_to_openai_function

In [ ]:
# 把 pydantic 模型自动转换成 OpenAI function calling 需要的 JSON Schema 格式
# （等价于我们在 L1 里手写的那种 functions 列表元素）
weather_function = convert_pydantic_to_openai_function(WeatherSearch)

In [ ]:
# 查看转换结果：包含 name / description / parameters，跟 L1 里手写的 JSON Schema 结构一致
weather_function

In [ ]:
# 缺少 docstring 会发生什么？—— description 会变成空字符串（不会报错，但信息缺失，模型可能更难判断何时调用它）
class WeatherSearch1(BaseModel):
    airport_code: str = Field(description="airport code to get weather for")

In [ ]:
convert_pydantic_to_openai_function(WeatherSearch1)

In [ ]:
# 缺少 Field(description=...) 会发生什么？—— 参数本身还是能被识别，只是参数的 description 会缺失
class WeatherSearch2(BaseModel):
    """Call this with an airport code to get the weather at that airport"""
    airport_code: str

In [ ]:
convert_pydantic_to_openai_function(WeatherSearch2)

In [ ]:
# 【版本兼容修复】langchain.chat_models 已经不再导出 ChatOpenAI，具体模型集成搬到了 langchain_openai
from langchain_openai import ChatOpenAI

In [ ]:
model = ChatOpenAI()

In [ ]:
# invoke() 时也可以临时传入 functions 参数（不用提前 bind），只对这一次调用生效
model.invoke("what is the weather in SF today?", functions=[weather_function])

In [ ]:
# 用 .bind() 把 functions 固定绑定到模型上，生成一个新的 Runnable，之后调用不用每次都传 functions
model_with_function = model.bind(functions=[weather_function])

In [ ]:
model_with_function.invoke("what is the weather in sf?")

## Forcing it to use a function

In [ ]:
# function_call={"name": "WeatherSearch"}：强制模型必须调用 WeatherSearch 这个函数
model_with_forced_function = model.bind(functions=[weather_function], function_call={"name":"WeatherSearch"})

In [ ]:
model_with_forced_function.invoke("what is the weather in sf?")

In [ ]:
# 即使输入跟天气毫不相关，只要强制了 function_call，模型也必须"硬调用" WeatherSearch
model_with_forced_function.invoke("hi!")

In [ ]:
# 【版本兼容修复】langchain.prompts 已不存在，ChatPromptTemplate 搬到了 langchain_core.prompts
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# system 消息设定模型的角色/行为，{input} 是用户输入的占位符
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "{input}")
])

In [ ]:
# 用 LCEL 把 prompt 和"已绑定 functions 的模型"串起来，组成一条完整的 chain
chain = prompt | model_with_function

In [ ]:
chain.invoke({"input": "what is the weather in sf?"})

## Using multiple functions

In [ ]:
# 再定义一个跟天气无关的函数：查询某个歌手的歌曲，用来测试模型在多个函数之间做选择
class ArtistSearch(BaseModel):
    """Call this to get the names of songs by a particular artist"""
    artist_name: str = Field(description="name of artist to look up")
    n: int = Field(description="number of results")

In [ ]:
# 把两个 pydantic 模型都转换成 OpenAI function schema，组成一个函数列表
functions = [
    convert_pydantic_to_openai_function(WeatherSearch),
    convert_pydantic_to_openai_function(ArtistSearch),
]

In [ ]:
model_with_functions = model.bind(functions=functions)

In [ ]:
# 期望：模型会自己判断出应该调用 WeatherSearch
model_with_functions.invoke("what is the weather in sf?")

In [ ]:
# 期望：模型会自己判断出应该调用 ArtistSearch，并且提取出 artist_name="taylor swift"、n=3
model_with_functions.invoke("what are three songs by taylor swift?")

In [ ]:
# 期望：跟两个函数都不相关时，模型选择直接用文字回复，不调用任何函数（function_call 没有被强制）
model_with_functions.invoke("hi!")